In [52]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from data_loader import load_options_data
from implied_vol import implied_volatility
from greeks import delta, gamma, vega, theta

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "spy_options_sample.csv"

df = load_options_data(DATA_PATH)

df["implied_vol"] = df.apply(
    lambda row: implied_volatility(
        market_price=row["mid_price"],
        S=row["underlying_price"],
        K=row["strike"],
        T=row["time_to_expiration"],
        r=row["risk_free_rate"],
        option_type=row["option_type"],
        q=0.0,
    ),
    axis=1,
)

In [53]:
df["delta"] = delta(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"], df["option_type"])
df["gamma"] = gamma(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"])
df["vega"] = vega(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"])
df["theta"] = theta(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"], df["option_type"])

df[["strike", "option_type", "mid_price", "implied_vol", "delta", "gamma", "vega", "theta"]]

,strike,option_type,mid_price,implied_vol,delta,gamma,vega,theta
0,580,C,25.00,0.133877,0.824566,0.010234,0.486497,-0.148373
1,590,C,17.10,0.127595,0.708916,0.014262,0.646140,-0.164838
2,600,C,10.60,0.080361,0.599212,0.019145,0.971160,-0.103990
3,610,C,6.15,0.061351,0.445022,0.021041,1.209557,-0.071218
4,620,C,3.45,0.067469,0.275876,0.016182,1.022977,-0.056308
5,580,P,4.75,0.120018,-0.219000,0.008039,0.903980,-0.040317
6,590,P,7.40,0.109096,-0.295785,0.009018,1.212979,-0.030140
7,600,P,10.30,0.099004,-0.370884,0.009730,1.482176,-0.018327
8,610,P,14.90,0.100461,-0.470504,0.010096,1.560587,-0.013608
9,620,P,20.50,0.102044,-0.567440,0.009824,1.542453,-0.005946


In [54]:
df["position"] = [10, 5, -8, -4, 3, -6, 7, 5, -3, 2]
df["contract_multiplier"] = 100

df["position_delta"] = df["position"] * df["contract_multiplier"] * df["delta"]
df["position_vega"] = df["position"] * df["contract_multiplier"] * df["vega"]
df["position_gamma"] = df["position"] * df["contract_multiplier"] * df["gamma"]
df["position_theta"] = df["position"] * df["contract_multiplier"] * df["theta"]

portfolio_exposure = df[[
    "position_delta",
    "position_vega",
    "position_gamma",
    "position_theta"
]].sum()

portfolio_exposure

position_delta    370.979301
position_vega     743.809112
position_gamma      3.778021
position_theta   -139.183820
dtype: float64

In [55]:
from quote_engine import generate_quotes

net_delta = portfolio_exposure["position_delta"]
net_vega = portfolio_exposure["position_vega"]

df["fair_value"] = df["mid_price"]

quotes = df.apply(
    lambda row: generate_quotes(
        fair_value=row["fair_value"],
        market_spread=row["bid_ask_spread"],
        implied_vol=row["implied_vol"],
        inventory=row["position"],
        net_delta=net_delta,
        net_vega=net_vega,
    ),
    axis=1,
)

quotes_df = quotes.apply(lambda x: pd.Series(x))

df = pd.concat([df, quotes_df], axis=1)

df[[
    "strike",
    "option_type",
    "position",
    "mid_price",
    "fair_value",
    "bid",
    "ask",
    "quoted_spread",
    "quote_skew"
]]

,strike,option_type,position,mid_price,fair_value,bid,bid,ask,ask,quoted_spread,quote_skew
0,580,C,10,25.00,25.00,24.8,24.692242,25.2,24.959181,0.266939,0.174288
1,590,C,5,17.10,17.10,16.9,16.843813,17.3,17.107610,0.263798,0.124288
2,600,C,-8,10.60,10.60,10.4,10.485621,10.8,10.725802,0.240180,-0.005712
3,610,C,-4,6.15,6.15,6.0,6.012874,6.3,6.218549,0.205676,0.034288
4,620,C,3,3.45,3.45,3.3,3.241344,3.6,3.450079,0.208735,0.104288
5,580,P,-6,4.75,4.75,4.6,4.618207,4.9,4.853216,0.235009,0.014288
6,590,P,7,7.40,7.40,7.2,7.128438,7.6,7.382986,0.254548,0.144288
7,600,P,5,10.30,10.30,10.1,10.050961,10.5,10.300463,0.249502,0.124288
8,610,P,-3,14.90,14.90,14.7,14.730596,15.1,14.980827,0.250230,0.044288
9,620,P,2,20.50,20.50,20.2,20.255201,20.8,20.556223,0.301022,0.094288


In [56]:
from fill_sim import simulate_fill
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

fills = df.apply(
    lambda row: simulate_fill(
        quoted_bid=row["bid"].iloc[-1] if hasattr(row["bid"], "iloc") else row["bid"],
        quoted_ask=row["ask"].iloc[-1] if hasattr(row["ask"], "iloc") else row["ask"],
        fair_value=row["fair_value"],
        quoted_spread=row["quoted_spread"],
        market_spread=row["bid_ask_spread"],
        volume=row["volume"],
        rng=rng,
    ),
    axis=1,
)

fills_df = fills.apply(pd.Series)

df = pd.concat([df, fills_df], axis=1)

df[[
    "strike",
    "option_type",
    "fair_value",
    "quoted_spread",
    "fill_probability",
    "filled",
    "fill_side",
    "fill_price",
    "spread_capture"
]]

,strike,option_type,fair_value,quoted_spread,fill_probability,filled,fill_side,fill_price,spread_capture
0,580,C,25.00,0.266939,0.562988,False,None,NaN,0.000000
1,590,C,17.10,0.263798,0.577091,True,buy,16.843813,0.256187
2,600,C,10.60,0.240180,0.647640,False,None,NaN,0.000000
3,610,C,6.15,0.205676,0.560782,True,sell,6.218549,0.068549
4,620,C,3.45,0.208735,0.533730,False,None,NaN,0.000000
5,580,P,4.75,0.235009,0.475279,False,None,NaN,0.000000
6,590,P,7.40,0.254548,0.601776,False,None,NaN,0.000000
7,600,P,10.30,0.249502,0.628206,True,sell,10.300463,0.000463
8,610,P,14.90,0.250230,0.609474,True,buy,14.730596,0.169404
9,620,P,20.50,0.301022,0.709645,False,None,NaN,0.000000


In [57]:
from inventory_manager import InventoryManager

inventory = InventoryManager(contract_multiplier=100)

inventory.process_fill(
    symbol="SPY_20260220_600_C",
    fill_side="buy",
    fill_price=10.50,
    quantity=1,
)

inventory.process_fill(
    symbol="SPY_20260220_600_C",
    fill_side="sell",
    fill_price=10.80,
    quantity=1,
)

inventory.get_positions(), inventory.get_cash(), inventory.get_trade_log()

({'SPY_20260220_600_C': 0},
 30.0,
 [{'symbol': 'SPY_20260220_600_C',
   'side': 'buy',
   'quantity': 1,
   'fill_price': 10.5,
   'cash_change': -1050.0,
   'new_position': 1,
   'cash': -1050.0},
  {'symbol': 'SPY_20260220_600_C',
   'side': 'sell',
   'quantity': 1,
   'fill_price': 10.8,
   'cash_change': 1080.0,
   'new_position': 0,
   'cash': 30.0}])